# Post-processing of recorded spectra

Load the on/off spectrum pairs recorded by `rtlsdr-recorder`, clean and
downsample them, average them, and export the result to a FITS spectrum.

The recorder saves frequency-switched pairs of one-second integrations: an
*on* spectrum centered on the HI line at 1420 MHz and an *off* spectrum
centered 4 MHz below, so that the two bands do not overlap and the off band
is free of Galactic HI emission.

The difference spectrum can be reduced in two ways, which are described in the two sections below respectively.
In each section, there are three sub-sections which show:
* An outline of the algorithm
* The step-by-step reduction of the data following the algorithm
* An example of how the same steps can be run with a single convenience ``reduce_spectra`` function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rtlsdr_recorder import frequency_array
from rtlsdr_recorder.analysis import (
    load_spectrum_pairs,
    clean_spectrum,
    downsample_spectrum,
    average_spectra,
    plot_spectrum_pair,
    reduce_spectra,
    write_fits,
)

In [ ]:
OUTPUT_DIRECTORY = '../lna-cable-rtl-usb-wifi-off/'

## Reduction with `clip_difference=True` (the default)

### Algorithm

1. Load all matched on/off spectrum pairs from the data directory.
2. For each pair, subtract the off spectrum from the on spectrum: since
   both were measured through the same receiver channels moments apart,
   this removes the bandpass shape and leaves the HI signal, even if the
   bandpass changes over the course of the observation.
3. Mask RFI spikes in each difference spectrum by iterative sigma clipping.
4. Mask the channels at the center of the band, which always contain a
   spurious DC spike from the receiver.
5. Spectrally downsample each difference spectrum.
6. Average all the difference spectra, ignoring masked values (channels
   masked in every spectrum, such as the DC region, become NaN).

### Step by step

Load all matched on/off spectrum pairs:

In [ ]:
spectra_on, spectra_off = load_spectrum_pairs(OUTPUT_DIRECTORY)
len(spectra_on)

Plot the first pair and its difference:

In [ ]:
plot_spectrum_pair(spectra_on[0], spectra_off[0], lw=0.1);

Form the per-pair differences, removing the bandpass shape, and mask RFI
spikes and the central DC spike with `clean_spectrum`:

In [ ]:
diffs = [clean_spectrum(on - off) for on, off in zip(spectra_on, spectra_off)]

plt.figure(figsize=(6,2))
plt.plot(frequency_array(), diffs[0], lw=0.1)
plt.xlabel('Frequency (MHz)');

Spectrally downsample the cleaned difference spectra:

In [ ]:
DOWNSAMPLE = 10
freq = downsample_spectrum(frequency_array(), DOWNSAMPLE)
diffs = [downsample_spectrum(diff, DOWNSAMPLE) for diff in diffs]

plt.figure(figsize=(6,2))
plt.plot(freq, diffs[0])
plt.xlabel('Frequency (MHz)');

That was the first pair only, so now average the differences of all pairs:

In [ ]:
average_diff = average_spectra(diffs)

plt.figure(figsize=(6,2))
plt.plot(freq, average_diff)
plt.xlabel('Frequency (MHz)');

### The same reduction with `reduce_spectra`

The single call gives the same frequencies, averages, and difference
spectrum as the steps above:

In [ ]:
reduced = reduce_spectra(OUTPUT_DIRECTORY)
np.testing.assert_allclose(reduced.frequencies, freq)
np.testing.assert_allclose(reduced.spectrum_diff, average_diff)

## Reduction with `clip_difference=False`

### Algorithm

1. Load all matched on/off spectrum pairs as above.
2. Mask RFI spikes in each on and each off spectrum by iterative sigma
   clipping.
3. Mask the channels at the center of each band as above.
4. Spectrally downsample each spectrum.
5. Average all the on spectra and all the off spectra, ignoring masked
   values.
6. Subtract the averaged off spectrum from the averaged on spectrum: this
   removes the bandpass shape and leaves the HI signal.

The averaged on and off spectra returned by `reduce_spectra` are always
computed with these steps; the flag only changes how the difference
spectrum is derived.

### Step by step

Clean the on and off spectra individually, downsample, average each set,
and subtract:

In [ ]:
spectra_on = [downsample_spectrum(clean_spectrum(spec), DOWNSAMPLE) for spec in spectra_on]
spectra_off = [downsample_spectrum(clean_spectrum(spec), DOWNSAMPLE) for spec in spectra_off]
average_on = average_spectra(spectra_on)
average_off = average_spectra(spectra_off)

plot_spectrum_pair(average_on, average_off, frequencies=freq);

### The same reduction with `reduce_spectra`

The single call with `clip_difference=False` gives the same on and off
averages and difference spectrum as the steps above:

In [ ]:
reduced_individual = reduce_spectra(OUTPUT_DIRECTORY, clip_difference=False)
np.testing.assert_allclose(reduced_individual.spectrum_on, average_on)
np.testing.assert_allclose(reduced_individual.spectrum_off, average_off)
np.testing.assert_allclose(reduced_individual.spectrum_diff, average_on - average_off)

## Export

Export the averaged difference spectrum from the default reduction to a
FITS file:

In [ ]:
write_fits(average_diff, 'spectrum.fits', frequencies=freq, overwrite=True)